# 05b: Does neutron-free p+d open a window at keV screening?

Notebook 05 found **no window** for p+d up to 575 eV: p+d never reached watt level, and coexisting d+d pairs crossed the neutron limit first. Since then, Karahadian et al. (2026) reported screening of **0.9 keV (TiD) and 1.7 keV (PdD)**. This is the one place the paper's conclusions could still change: at keV screening, p+d might reach watts while the d+d pairs in the same lattice stay below the 1990 neutron limit.

**Test:** for mixed H/D loading with deuterium fraction $x_D$,
* p+d power ∝ $x_D(1-x_D)\,\Gamma_{pd}$ must reach watt level (Q = 5.49 MeV), and
* d+d neutrons ∝ $x_D^2\,\Gamma_{dd}$ must stay below 2×10⁻²⁶ per pair-site.

Both reactions use the same beam-matched screening profile (screening depends on charge, which is the same). Scanned over U_meas = 0.9, 1.7 and 2.0 keV, four profile shapes and spacings 2.75 and 0.94 Å.

In [1]:
import numpy as np, matplotlib.pyplot as plt
from cfsolve import *
plt.rcParams.update({"axes.spines.top":False,"axes.spines.right":False,"axes.grid":True,"grid.alpha":.25,"font.size":10})
NU0 = 37e-3/4.135667e-15; LL = np.log10(2e-26)
need_pd = np.log10(5.6e-11*3.65/5.49)
ELLS = np.logspace(np.log10(0.002), np.log10(0.4), 48)
def band(pair, d0A, U, shape):
    a, b = pair; mu = reduced_mass(a, b); S = 2.0e-4 if pair == ("p", "d") else S_KEVB
    Vb = V_lattice_pair(a, b, d0_A=d0A); d0 = d0A/BOHR_A; Rmax = d0 + 7; Rmin = 0.4*d0
    E_lat = solve(Vb, Rmax, Rmin_well=Rmin, mu=mu, S_keVb=S)["E"]; out = []
    for l in ELLS:
        ex = extra_screening(beam_matched_dU(U, l, shape), l, shape); V = lambda R: Vb(R) + ex(R)
        try: out.append(solve(V, Rmax, Rmin_well=Rmin, mu=mu, S_keVb=S)["log10_rate"])
        except Collapse:
            B = solve(V, Rmax, Rmin_well=0.0, mu=mu, S_keVb=S); r = np.logspace(-4, np.log10(Rmax), 200000); v = V(r)
            imin = np.argmin(np.where(r < Rmin, v, np.inf)); sel = (r > r[imin]) & (r < d0) & (v > E_lat)
            G = 2*np.trapezoid(np.sqrt(2*mu*np.clip(v[sel]-E_lat, 0, None)), r[sel])
            out.append(min(np.log10(NU0) - G/np.log(10), B["log10_rate"]))
    return np.array(out)
xD = np.logspace(-8, np.log10(0.5), 600)

In [2]:
rows = []
for U in [900, 1700, 2000]:
    for d in [2.75, 0.94]:
        for sh in ["exponential", "gaussian", "step", "yukawa"]:
            pd, dd = band(("p", "d"), d, U, sh), band(("d", "d"), d, U, sh)
            win = None
            for j, ell in enumerate(ELLS):
                ok = (np.log10(xD*(1 - xD)) + pd[j] >= need_pd) & (2*np.log10(xD) + dd[j] < LL)
                if ok.any():
                    win = (ell, xD[ok].min(), xD[ok].max(), pd[j], dd[j]); break
            xs = np.minimum(0.5, 10**((LL - dd)/2)); short = need_pd - (pd + np.log10(xs*(1 - xs)))
            jb = np.argmin(short); rows.append((U, d, sh, pd.max(), dd.max(), win, short[jb], ELLS[jb], xs[jb]))
            w = (f"WINDOW at ell = {win[0]:.4f} A, x_D {win[1]:.1e}–{win[2]:.1e} (p+d 10^{win[3]:.1f}, d+d 10^{win[4]:.1f})"
                 if win else "no window")
            print(f"U = {U:4d} eV, d = {d:4.2f} A, {sh:11s}: max p+d 10^{pd.max():6.1f}, max d+d 10^{dd.max():6.1f}  ->  {w}; closest: {short[jb]:.1f} orders short (ell {ELLS[jb]:.3f} A, x_D {xs[jb]:.1e})")

U =  900 eV, d = 2.75 A, exponential: max p+d 10^  -9.0, max d+d 10^  -8.0  ->  no window; closest: 7.0 orders short (ell 0.162 A, x_D 3.8e-09)


U =  900 eV, d = 2.75 A, gaussian   : max p+d 10^ -15.5, max d+d 10^ -16.5  ->  no window; closest: 8.4 orders short (ell 0.319 A, x_D 5.0e-04)


U =  900 eV, d = 2.75 A, step       : max p+d 10^ -28.9, max d+d 10^ -32.3  ->  no window; closest: 19.1 orders short (ell 0.319 A, x_D 5.0e-01)


U =  900 eV, d = 2.75 A, yukawa     : max p+d 10^  -6.3, max d+d 10^  -4.0  ->  no window; closest: 6.7 orders short (ell 0.021 A, x_D 1.5e-11)


U =  900 eV, d = 0.94 A, exponential: max p+d 10^  -7.3, max d+d 10^  -5.8  ->  no window; closest: 3.6 orders short (ell 0.092 A, x_D 1.8e-07)


...

U = 2000 eV, d = 2.75 A, yukawa     : max p+d 10^   0.1, max d+d 10^   3.8  ->  no window; closest: 4.2 orders short (ell 0.008 A, x_D 1.8e-15)


U = 2000 eV, d = 0.94 A, exponential: max p+d 10^  -3.0, max d+d 10^  -0.5  ->  no window; closest: 2.8 orders short (ell 0.083 A, x_D 9.7e-09)


U = 2000 eV, d = 0.94 A, gaussian   : max p+d 10^  -8.1, max d+d 10^  -8.1  ->  no window; closest: 4.4 orders short (ell 0.182 A, x_D 2.3e-07)


U = 2000 eV, d = 0.94 A, step       : max p+d 10^ -19.0, max d+d 10^ -21.5  ->  no window; closest: 9.2 orders short (ell 0.182 A, x_D 5.0e-01)


U = 2000 eV, d = 0.94 A, yukawa     : max p+d 10^   0.1, max d+d 10^   3.8  ->  no window; closest: 4.2 orders short (ell 0.008 A, x_D 1.8e-15)


In [3]:
nw = sum(r[5] is not None for r in rows)
print(f"variants: {len(rows)}; with a window: {nw}")
reach = [r for r in rows if r[3] + np.log10(0.25) >= need_pd]
best = min(rows, key=lambda r: r[6])
print(f"closest approach over all variants: p+d {best[6]:.1f} orders short of watts at the neutron-allowed x_D ({best[0]} eV, {best[1]} A, {best[2]}, ell {best[7]:.3f} A, x_D {best[8]:.1e})")
print(f"variants where p+d alone reaches watts (x_D = 0.5): {len(reach)}")
for r in reach:
    print(f"   U={r[0]}, d={r[1]}, {r[2]}: p+d max 10^{r[3]:.1f}, d+d max 10^{r[4]:.1f}, window: {'yes' if r[5] else 'no'}")

variants: 24; with a window: 0
closest approach over all variants: p+d 2.8 orders short of watts at the neutron-allowed x_D (2000 eV, 0.94 A, exponential, ell 0.083 A, x_D 9.7e-09)
variants where p+d alone reaches watts (x_D = 0.5): 14
   U=900, d=2.75, exponential: p+d max 10^-9.0, d+d max 10^-8.0, window: no
   U=900, d=2.75, yukawa: p+d max 10^-6.3, d+d max 10^-4.0, window: no
   U=900, d=0.94, exponential: p+d max 10^-7.3, d+d max 10^-5.8, window: no
   U=900, d=0.94, yukawa: p+d max 10^-6.3, d+d max 10^-4.0, window: no
   U=1700, d=2.75, exponential: p+d max 10^-5.7, d+d max 10^-3.9, window: no
   U=1700, d=2.75, yukawa: p+d max 10^-1.0, d+d max 10^2.4, window: no
   U=1700, d=0.94, exponential: p+d max 10^-3.7, d+d max 10^-1.4, window: no
   U=1700, d=0.94, gaussian: p+d max 10^-8.7, d+d max 10^-8.6, window: no
   U=1700, d=0.94, yukawa: p+d max 10^-1.0, d+d max 10^2.4, window: no
   U=2000, d=2.75, exponential: p+d max 10^-4.9, d+d max 10^-2.8, window: no
   U=2000, d=2.75, yuka

## Interpretation

Numbers from the printed results above.

* **No window in any of the 24 variants** (U_meas = 0.9–2.0 keV, four shapes, spacings 2.75 and 0.94 Å).
* In 14 variants p+d *alone* would reach watt level in a 50/50 H/D mix (up to 10^0.1 s⁻¹ per pair for Yukawa at 2 keV). In each of them the peak d+d rate equals or exceeds the peak p+d rate (by 0–3.7 orders), mainly because the d+d S-factor is ~5×10⁵ times larger.
* Keeping the d+d neutrons below 2×10⁻²⁶ per pair-site forces the deuterium fraction down to 10⁻¹⁵–10⁻⁷. That leaves p+d **at least 2.8 orders short of watts** (closest case: 2 keV, 0.94 Å, exponential, ℓ = 0.083 Å, x_D ≈ 10⁻⁸), and 3.3–19 orders short in the others.
* This closes the paper's limitation (v): **even neutron-free p+d cannot deliver watts at rest without coexisting d+d pairs breaking the 1990 neutron limit, up to the highest reported screening (1.7 keV) and beyond (2 keV).** The model assumes random H/D occupancy. The only escape is a lattice that pairs each deuteron with a proton far more often than with another deuteron: enough to recover at least 2.8 orders, i.e. d–d pairs suppressed ≳10^5.6 (≈4×10⁵)-fold below random mixing at the required p+d pair count (raising x_D by 10^2.8 raises random d–d pairing by 10^5.6). Experiment E1 (a gamma search on H/D co-loaded Pd) tests the p+d channel directly, whatever the pairing.
